In [ ]:
from bctools.io import InstrumentResponse
from bctools.loc import LocalLocTable
from bctools.spectra.spectrum import BandFunction,Comptonized
import os
import math
import multiprocessing
import itertools
import pickle 
import math
import numpy as np
import itertools
from bctools.loc import TSMap, NormLocLike
import astropy.units as u
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
import healpy as hp

run_name = "run10"
irf_path = "/data/models/irf_summed_"+run_name+".h5"
dir_path = "/data/test_newrepo/"

## The plot is generated using the mean spectrum dataset and mean spectrum LUT.

In [ ]:
load_from_file = True

# The code is inspired by the bc-tools tutorial.
if load_from_file:

    with open('/data/models/LUTS/medium_lut_' + run_name + '.pkl', 'rb') as f:
        medium_sky_loctable = pickle.load(f)
    
else:

    with InstrumentResponse(irf_path) as irf: 
        
        # Hypothetical spectrum
        # This normalization corresponds to 1 ph/cm2/s between 50-300 keV
        #spectrum = PowerLaw(60,2)
        #spectrum = PowerLaw._from_megalib(['PowerLaw',10,10000,2],"5.0")

        medium_spectrum = BandFunction._from_megalib(['BandFunction',10,10000,-1,-2.3,699.9],"10.0")
      
        # In this case we integrate the rate from all energy channels. 
        # You can subdivide the data into multiple energy channel groups

        medium_local_loctable = LocalLocTable.from_irf(irf, medium_spectrum,energy_channels = 1)
   
        
    # The local_loctable contains the expected rates in spacecraft coordinates
    # We now need to use this to estimate the total expected counts in sky coordinate for
    # the full duration of an event. 
    # In this case we simply have a 1 second event and specifying the attitude by a quaternion
    # ([0,0,0,1] corresponds to the identity rotation). You can have multiple attitude-duration
    # pairs to correctly model long duration events.
    medium_sky_loctable = medium_local_loctable.to_skyloctable(attitude = [0,0,0,1], duration = 1)
    
    #Store LUT
    
    # Salvataggio su file
    with open('/data/models/LUTS/medium_lut_'+run_name+'.pkl', 'wb') as f:
        pickle.dump(medium_sky_loctable, f)


In [ ]:

def angular_distance(theta1, phi1, theta2, phi2):
    """
    Compute the angular (great-circle) distance in degrees between two points
    specified by (theta, phi) coordinates in degrees.

    Parameters:
        theta1, phi1: floats, coordinates of the first point (degrees).
        theta2, phi2: floats, coordinates of the second point (degrees).

    Returns:
        angular_dist_deg: float, angular distance between the two points in degrees.
    """
    # Adjust theta values by shifting -90
    theta1 = 90 - theta1
    theta2 = 90 - theta2

    # Convert angles to radians
    theta1_rad = math.radians(theta1)
    phi1_rad = math.radians(phi1)
    theta2_rad = math.radians(theta2)
    phi2_rad = math.radians(phi2)
    
    # Compute the difference in longitude
    delta_phi = abs(phi1_rad - phi2_rad)

    # Apply spherical law of cosines
    value = (math.sin(theta1_rad) * math.sin(theta2_rad) +
             math.cos(theta1_rad) * math.cos(theta2_rad) * math.cos(delta_phi))
    
    # Clamp value to [-1, 1] to avoid floating point errors in acos
    value_clamped = max(-1.0, min(1.0, value)) 
    
    angular_dist = math.acos(value_clamped)
    
    # Convert angular distance from radians to degrees
    angular_dist_deg = math.degrees(angular_dist)
    
    return angular_dist_deg

In [ ]:
print(f"Medium Look-up tables: {medium_sky_loctable.labels}")

In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').data

In [ ]:
import pickle
if False:

    # Salva l'array in un file usando pickle
    with open("/data/exp_medium_Y1.pkl", "wb") as f:
        pickle.dump(medium_sky_loctable.get_expectation_map('BGO_Y1').data, f)


In [ ]:
import healpy as hp
from astropy.coordinates import SkyCoord
import astropy.units as u
import numpy as np

def get_coord_helpix(nside, pix_id):
    
    # Parameters
    nside = 16  # Replace with your NSIDE value
    ipix = 1    # Replace with the HEALPix pixel ID (nested scheme)
    
    # Get the angular coordinates (theta, phi) of the pixel center
    theta, phi = hp.pix2ang(nside, ipix, nest=True)
    
    # Convert to equatorial coordinates (RA, Dec)
    ra = phi * 180.0 / np.pi            # phi is longitude in radians (RA)
    dec = 90.0 - theta * 180.0 / np.pi  # theta is colatitude (Dec)
    
    # Create the SkyCoord object
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')

    return coord


In [ ]:
medium_sky_loctable.get_expectation_map('BGO_Z1').plot()

In [ ]:
#test with LUTs
import pickle 
shared = True
numpy_file = 0

run_name_test = "dataset"
file_name_test = "run57_mix_mega_shared"

file_path ='/data/analysis/'+run_name_test+'/'+file_name_test+'_dataset.pkl'

# Load the array from the pickle file
with open(file_path, 'rb') as file:
    loaded_array_test = pickle.load(file)


In [ ]:
loaded_array_test.shape

In [ ]:
import numpy as np
counts_array = []
for grb in loaded_array_test:
    counts_array.append(grb['counts'])
counts_array_np = np.array(counts_array)
print(counts_array_np.shape)

In [ ]:
np.sum(counts_array_np[:,0])

In [ ]:
plt.hist(counts_array_np[:,0])
plt.title("Counts Hist. for Z1 panel")

In [ ]:
import numpy as np

filter_flux = 0
filter_spectra = 0

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_flux == 1:
    for grb in loaded_array_test:
        if grb['flux'] > 10 and grb['flux'] <= 15:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

filtered_loaded_array_test = np.empty(loaded_array_test.shape[0], dtype=object)

count = 0

if filter_spectra==1:
    for grb in loaded_array_test:
        #if  grb['spectra'] == 'medium':
        if  '1500' in grb['spectrum']:
            filtered_loaded_array_test[count] = grb
            count = count+1

    loaded_array_test = filtered_loaded_array_test[:count]

In [ ]:
loaded_array_test.shape
test_dataset=loaded_array_test

In [ ]:
b_sim = np.array([57.6053, 58.7157, 51.4131, 48.2891, 47.7293, 45.9617])

def process_lut_source(grb):
    
    theta_real = float(grb['coord'][0])
    phi_real = float(grb['coord'][1])
    counts = grb['counts']
    s_counts = np.array([grb['counts'][3],grb['counts'][2],grb['counts'][5],grb['counts'][4],grb['counts'][1],grb['counts'][0]])
    b_counts = [1,1,1,1,1,1]
    spectra_value = grb['spectrum']

    coord_grb = grb['coord']

    
    ra = float(coord_grb[1])
    dec = 90.0 - float(coord_grb[0])
    
    
    coord = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame='icrs')

    #test LUT
    # The source counts are the sum of the expected counts in the LUT and the background counts.
    s_counts = medium_sky_loctable.get_expectation(coord) + np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20
    b_counts =  np.array([b_sim[3],b_sim[2],b_sim[5],b_sim[4],b_sim[1],b_sim[0]])*20
    medium_sqrt_ts, mdium_ra_loc, medium_dec_loc, medium_cont_radius,medium_ts,medium_loc_tsvalue,medium_cont_area = localize_grb(medium_sky_loctable,s_counts, b_counts,theta_real,phi_real)
    
    max_ts = medium_sqrt_ts
   
    if max_ts == medium_sqrt_ts:
        original_ts_value = medium_loc_tsvalue
        best_ts = medium_ts
        ra_loc=mdium_ra_loc
        dec_loc=medium_dec_loc
        cont_radius = medium_cont_radius
        cont_area = medium_cont_area
   
    theta_loc, phi_loc = ra_dec_to_theta_phi(ra_loc, dec_loc)

    dist = angular_distance(theta_loc, phi_loc, theta_real, phi_real)
    theta_dist = np.abs(theta_loc - theta_real)
    phi_dist = diff_phi(phi_loc, phi_real)

    result = [theta_real, phi_real, theta_loc, phi_loc, dist, theta_dist, phi_dist, max_ts, cont_radius,best_ts,original_ts_value,cont_area]
    
    return result

def localize_grb(sky_table, signal, background, theta_deg, phi_deg):
    # Set background and signal data into the sky table
    sky_table.set_background(background)
    sky_table.set_data(signal)

    # Define a TS map with nside = 32 (finer resolution than the lookup table)
    ts_map = TSMap(nside=32, coordsys='icrs')

    # Create a normalized likelihood (Poisson likelihood for counting instruments)
    # The only free parameter is the overall normalization
    norm_likelihood = NormLocLike(sky_table)
   
    # Compute the TS map from the likelihood
    ts_map.compute(norm_likelihood)
    
    # Validate theta before converting to radians
    if theta_deg < 0:
        print(f"Invalid theta value: {theta_deg}")
        theta_deg = 0
    elif theta_deg > 180:
        print(f"Invalid theta value: {theta_deg}")
        theta_deg = 180
    
    # Convert degrees → radians
    theta_rad = np.deg2rad(theta_deg)
    phi_rad = np.deg2rad(phi_deg)
    
    # Check if theta is in a valid radian range
    if theta_rad < 0 or theta_rad > np.pi:
        print(f"theta_rad out of range: {theta_rad}")

    # Get the pixel index corresponding to the given (theta, phi) coordinates
    pixel_index = ts_map.ang2pix(theta_rad, phi_rad)

    # Extract the TS value at the given pixel
    ts_value_at_coords = ts_map._data[pixel_index]

    # Define an array of confidence levels (0% → 100%)
    confidence_levels = np.arange(0, 1.01, 0.01)
    
    # Compute containment radii for each confidence level
    containment_radii = np.array([
        np.sqrt(ts_map.error_area(cont=cont) / np.pi).to(u.deg).value
        for cont in confidence_levels
    ])
  
    # Compute containment area at 90% confidence
    containment_area_90 = ts_map.error_area(cont=0.9).to(u.deg**2).value

    # Return:
    # - maximum TS value
    # - RA and Dec of the best localization (in degrees)
    # - containment radii
    # - ts_map object
    # - TS value at input coordinates
    # - containment area at 90% confidence
    return (
        np.max(ts_map),
        ts_map.best_loc().ra.deg,
        ts_map.best_loc().dec.deg,
        containment_radii,
        ts_map,
        ts_value_at_coords,
        containment_area_90
    )


def ra_dec_to_theta_phi(ra, dec):

    theta = 90-dec
    phi = ra
    
    return theta, phi

def diff_phi(a1, a2):
    
    diff = abs(a1 - a2)
    if diff > 180:
        diff = 360 - diff
    
    return diff

In [ ]:
results_bkg = []
spectra_fitted = 0

def init_worker():
    seed = int.from_bytes(os.urandom(4), "little")
    np.random.seed(seed)

def process_in_parallel(test_dataset):
    with multiprocessing.Pool(processes=200, initializer=init_worker) as pool:
        results = pool.starmap(process_lut_source, zip(test_dataset))
    return results

results_bkg = process_in_parallel(test_dataset)


In [ ]:
distances = []
theta_distances = []
phi_distances = []
cont_radius_list = []
cont_area_list = []
for res in results_bkg:
    distances.append(res[4])
    theta_distances.append(res[5])
    phi_distances.append(res[6])
    cont_radius_list.append(res[8])
    cont_area_list.append(res[11])



In [ ]:

hp.projview(
    np.array(cont_area_list),
    coord=["G"],
    projection_type="aitoff",          # Cambiato da "mollweide" a "aitoff"
    graticule=True,
    graticule_labels=True,
    longitude_grid_spacing=60,
    title=file,
    latitude_grid_spacing=30,
    cmap="turbo",
    nest=True,
    unit="", 
    fontsize={
        "xlabel": 14,
        "ylabel": 14,
        "title": 16,
        "xtick_label": 14,
        "ytick_label": 14,
        "cbar_label": 14,
        "cbar_tick_label": 14  # qui imposti il font size dei numeri della colorbar
    },
    override_plot_properties={
        "cbar_shrink": 0.8,
        "cbar_pad": 0.05,
        "cbar_label_pad": 5
    }
    
)
plt.show()

In [ ]:
import pickle
if False:

    with open("/data/bc_"+file_name_test+"_distall_plot.pkl", "wb") as f:
        pickle.dump(distances, f)
        
    with open("/data/bc_"+file_name_test+"_cont_area_plot.pkl", "wb") as f:
        pickle.dump(cont_area_list, f)


In [ ]:
print(np.mean(distances))
print(np.mean(theta_distances))
print(np.mean(phi_distances))
print(np.mean(cont_area_list))